# Dimensionality Reduction with PCA and t-SNE on MNIST handwritten digit

Dimensionality is the number of dimensions, features, or variables associated with a sample of data.
Methods are commonly divided into linear and non-linear approaches. Approaches can also be divided into feature selection and feature extraction.

Applications of Dimensionality Reduction:
    1. feature engineering
    2. Noise reduction
    3. Generating plausible artificial datasets
    4. Financial modelling/risk analysis
    

In [ ]:
# Data Set : MNIST handwritten digit classification
url="https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/optdigits.tra"

The MNIST handwritten digit dataset is one of the most famous image recognition datasets. It consists of square images, 8 x 8 pixels, each containing a single handwritten digit. Thus, the dataset features are an 8 by 8 matrix, containing each pixel's color in grayscale. The target consists of 10 classes, one for each digit from 0 to 9.  32x32 bitmaps are divided into nonoverlapping blocks of 4x4 and the number of on pixels are counted in each block. This generates an input matrix of 8x8 where each element is an integer in the range 0..16.

In [ ]:
# toolboxes/libraries
import pandas as pd
import numpy as np

# Visualization libraries

import matplotlib.pyplot as plt 
import matplotlib.patheffects as PathEffects
import matplotlib
import seaborn as sns
sns.set(color_codes=True)

%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, Normalizer, MinMaxScaler

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
# from sklearn.decomposition import TruncatedSVD as SVD



In [ ]:
# import data
df =pd.read_csv(url, header=None)

In [ ]:
# check data
df.head()

In [ ]:
# show data type of each attributes
df.info()

In [ ]:
# rename columns
df.columns = ["X" + str(i) for i in range(0, len(df.columns)-1)] + ["class"]
df.columns

In [ ]:
df['class'].value_counts()

In [ ]:
sns.countplot(data=df, x="class")

In [ ]:
# Display images first 20 rows of last columns
fig, ax = plt.subplots(
    nrows=1,
    ncols=20,
    figsize=(16,4),
    subplot_kw=dict(xticks=[], yticks=[]) )

for i in np.arange(20):
    ax[i].imshow(df.to_numpy()[i,0:-1].reshape(8,8), cmap=plt.cm.gray ) # display last columns

### Splitting the Data-Set into Training Set and Test Set

In [ ]:
X = df.iloc[:,:-1]  # feature
y = df['class'] # target

# in this our main data is splitted into train and test

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=100)
# we can check their dimension
print(X_train.shape)
print(X_test.shape)

## Principal Component Analysis (PCA)

PCA is a projection technique which find a projection of the data in a smaller dimension. The idea is to find an axis in the data with highest variance and to map the data along that axis.

PCA is a method of obtaining important variables (in form of components) from a large set of variables available in a data set. It extracts low dimensional set of features by taking a projection of irrelevant dimensions from a high dimensional data set with a motive to capture as much information as possible.
https://www.analyticsvidhya.com/blog/2016/03/pca-practical-guide-principal-component-analysis-python/

In [ ]:
#  Perform PCA - principle component analysis
df_sc = StandardScaler().fit_transform(X_train)  # scale the features

pca = PCA()

df_pca = pca.fit_transform(df_sc)

plt.plot(pca.explained_variance_ratio_)

In [ ]:
# http://jmausolf.github.io/code/pca_in_python/
# Variance Explained
# Related to scree plots and eigenvalues is the variance explained (which is directly related to the eigenvalues). This can also be visualized.
def var_explained():
    import numpy as np
    from matplotlib.pyplot import figure, show
    from matplotlib.ticker import MaxNLocator

    ax = figure().gca()
    ax.plot(np.cumsum(pca.explained_variance_ratio_))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.axvline(x=32, linewidth=1, color='r', alpha=0.5)
    plt.title('Explained Variance of PCA by Component')
    show()

var_explained()

In [ ]:
print([(j, np.cumsum(pca.explained_variance_ratio_)[j]) for j in range(len(pca.explained_variance_ratio_[:33]))])

Here 32 components explain the almost 97% variance. So we can reduce the dimension according to the required variance.  Surprisingly, 4 components would explain almost 54% of the variance.

In [ ]:
# https://www.programmersought.com/article/84117014666/

# Display the first 2 dimensional spatial distribution of 10 types of handwritten digital pictures after PCA compression.
#Define the color of the scatter plot points, draw black points when i takes 0, and so on.

def plot_pca_scatter():

    colors = ['black', 'blue', 'purple', 'yellow', 'white', 'red', 'lime', 'cyan', 'orange', 'gray']
    for i in range(len(colors)):
        px = df_pca[:, 0][np.array(y_train) == i]
        py = df_pca[:, 1][np.array(y_train)== i]
        plt.scatter(px, py, c=colors[i])
    
    plt.legend(np.arange(0,10).astype(str))
    plt.xlabel('First Principal Component')
    plt.ylabel('Second Principal Component')
    plt.show()
    
plot_pca_scatter()

In [ ]:
# Display the next 2 dimensional spatial distribution
def plot_pca_scatter2():

    colors = ['black', 'blue', 'purple', 'yellow', 'white', 'red', 'lime', 'cyan', 'orange', 'gray']
    for i in range(len(colors)):
        px = df_pca[:, 2][np.array(y_train) == i]
        py = df_pca[:, 3][np.array(y_train)== i]
        plt.scatter(px, py, c=colors[i])
    
    plt.legend(np.arange(0,10).astype(str))
    plt.xlabel('Third Principal Component')
    plt.ylabel('Fourth Principal Component')
    plt.show()
    
plot_pca_scatter2()

### Limitations of PCA

PCA is a linear algorithm. It will not be able to interpret complex polynomial relationship between features. On the other hand, t-SNE is based on probability distributions with random walk on neighborhood graphs to find the structure within the data.

A major problem with, linear dimensionality reduction algorithms is that they concentrate on placing dissimilar data points far apart in a lower dimension representation. But in order to represent high dimension data on low dimension, non-linear manifold, it is important that similar datapoints must be represented close together, which is not what linear dimensionality reduction algorithms do.

https://www.analyticsvidhya.com/blog/2017/01/t-sne-implementation-r-python/

## t-Distributed Stochastic Neighbor Embedding (t-SNE)

t-SNE is a non-linear dimensionality reduction algorithm used for exploring high-dimensional data. It maps multi-dimensional data to lower dimensions which are easy to visualize.

It is extensively applied in Image processing, NLP, genomic data and speech processing. It has been utilized for improving the analysis of brain and heart scans

In [ ]:
# t-SNE stands for t-Distributed Stochastic Neighbor Embedding
df_nm = MinMaxScaler().fit_transform(X_train)  # scale the features

df_tsne = TSNE(n_components = 2,perplexity=50, random_state=1).fit_transform(df_nm)

In [ ]:
## Another Option to normalize values

# scaler = Normalizer().fit(X_train)
# df_tsne = scaler.transform(X_train)

In [ ]:
# https://www.oreilly.com/content/an-illustrated-introduction-to-the-t-sne-algorithm/
def scatter(x, colors):
    # We choose a color palette with seaborn.
    palette = np.array(sns.color_palette("hls", 10))

    # We create a scatter plot.
    f = plt.figure(figsize=(8, 8))
    ax = plt.subplot(aspect='equal')
    sc = ax.scatter(x[:,0], x[:,1], lw=0, s=40,
                    c=palette[colors.astype(np.int)])
    plt.xlim(-25, 25)
    plt.ylim(-25, 25)
    ax.axis('off')
    ax.axis('tight')

    # We add the labels for each digit.
    txts = []
    for i in range(10):
        # Position of each label.
        xtext, ytext = np.median(x[colors == i, :], axis=0)
        txt = ax.text(xtext, ytext, str(i), fontsize=24)
        txt.set_path_effects([
            PathEffects.Stroke(linewidth=5, foreground="w"),
            PathEffects.Normal()])
        txts.append(txt)

    return f, ax, sc, txts

In [ ]:
scatter(df_tsne, y_train)

In [ ]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install -U yellowbrick

from yellowbrick.text import TSNEVisualizer

# https://www.scikit-yb.org/en/latest/api/text/tsne.html
# Create the visualizer and draw the vectors
tsne = TSNEVisualizer()
tsne.fit(X_train, y_train)
tsne.show()

t-SNE outputs provide better results than PCA and other linear dimensionality reduction models. This is because a linear method such as classical scaling is not good at modeling curved manifolds. It focuses on preserving the distances between widely separated data points rather than on preserving the distances between nearby data points.
https://www.analyticsvidhya.com/blog/2017/01/t-sne-implementation-r-python/